# Dataset Exploration

In [5]:
# pip install pandas ipykernel
import os

import pandas as pd

# Reuse the existing dataset-path resolver instead of duplicating the
# kagglehub cache lookup here.
from download_dataset import find_cached_download

TOP_N = 20  # how many smallest/largest countries to show in tables

## Locate the dataset

In [6]:
dataset_root = find_cached_download()
if dataset_root is None:
    from download_dataset import main as download_dataset

    dataset_root = download_dataset()

# find_cached_download() gives the versioned dataset root, but the
# per-country folders live one level deeper inside a wrapper dir
# (e.g. compressed_dataset/). Find that wrapper generically instead of
# hardcoding its name: it's the directory with the most immediate
# subdirectories, i.e. the parent of the 124 country folders.
country_level_dir = max(os.walk(dataset_root), key=lambda entry: len(entry[1]))[0]

print(f"Dataset root:       {dataset_root}")
print(f"Country-level dir:   {country_level_dir}")

Dataset root:       /Users/awsare/.cache/kagglehub/datasets/ubitquitin/geolocation-geoguessr-images-50k/versions/1
Country-level dir:   /Users/awsare/.cache/kagglehub/datasets/ubitquitin/geolocation-geoguessr-images-50k/versions/1/compressed_dataset


## Count images per country

In [7]:
# Each country folder contains its .jpg files directly (no further
# nesting), so counting entries in each immediate subfolder gives the
# per-class image count.
countries = [
    entry.name for entry in os.scandir(country_level_dir) if entry.is_dir()
]
counts = {
    country: len(os.listdir(os.path.join(country_level_dir, country)))
    for country in countries
}

df = pd.DataFrame(
    {"country": counts.keys(), "image_count": counts.values()}
)
total_images = df["image_count"].sum()
# Each country's share of the full dataset — this is what drives class
# imbalance during training.
df["percentage"] = df["image_count"] / total_images * 100
df = df.sort_values("image_count", ascending=False).reset_index(drop=True)

print(f"{len(df)} countries, {total_images} images total")
df

124 countries, 49997 images total


,country,image_count,percentage
0,United States,12014,24.029442
1,Japan,3840,7.680461
2,France,3573,7.146429
3,United Kingdom,2484,4.968298
4,Brazil,2320,4.640278
...,...,...,...
119,Antarctica,1,0.002000
120,Mozambique,1,0.002000
121,Tanzania,1,0.002000
122,Belarus,1,0.002000


## Smallest / largest countries

In [8]:
largest = df.head(TOP_N)
smallest = df.tail(TOP_N).sort_values("image_count")

print(f"Top {TOP_N} largest countries by image count:")
display(largest)

print(f"Top {TOP_N} smallest countries by image count:")
display(smallest)

Top 20 largest countries by image count:


,country,image_count,percentage
0,United States,12014,24.029442
1,Japan,3840,7.680461
2,France,3573,7.146429
3,United Kingdom,2484,4.968298
4,Brazil,2320,4.640278
5,Russia,1761,3.522211
6,Australia,1704,3.408204
7,Canada,1382,2.764166
8,South Africa,1183,2.366142
9,Spain,1075,2.150129


Top 20 smallest countries by image count:


,country,image_count,percentage
113,South Sudan,1,0.002
121,Tanzania,1,0.002
120,Mozambique,1,0.002
119,Antarctica,1,0.002
118,Venezuela,1,0.002
117,South Georgia and South Sandwich Islands,1,0.002
116,Qatar,1,0.002
115,Svalbard and Jan Mayen,1,0.002
114,Pitcairn Islands,1,0.002
123,Gibraltar,1,0.002
